# 03bis — Agent IA 2 V2
## Reconnaissance environnementale : forêt par vecteur ou par vision

Ce notebook reprend la récupération OSM robuste de la version précédente, puis ajoute
un **véritable Agent IA 2**.

Trois méthodes sont comparées :

```text
Méthode 0 — référence SIG
Python / Shapely / INTERSECT

Méthode 1 — Agent IA vectoriel
trace GPX + polygones OSM simplifiés → LLM

Méthode 2 — Agent IA visuel
image route + forêts + repères de distance → modèle multimodal
```

La méthode `INTERSECT` sert uniquement de **référence expérimentale**.
Elle n'est jamais donnée à l'agent avant sa prédiction.

Le fichier final utilisé par le notebook 04 reste :

```text
data/output/terrain_forest_annotations.json
```

In [1]:
import hashlib, json, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

from pyproj import Transformer
from shapely.geometry import LineString, Polygon, Point
from shapely.ops import transform, unary_union, polygonize

## 1. Charger la route

In [2]:
route_path = Path("data/output/route_resampled.csv")
if not route_path.exists():
    raise FileNotFoundError("Exécute d'abord le notebook 02.")

route = pd.read_csv(route_path)
required = {"distance_m","lat","lon","elev_m"}
missing = required - set(route.columns)
if missing:
    raise ValueError(f"Colonnes manquantes : {sorted(missing)}")

print("Points :", len(route))
print("Longueur :", round(route["distance_m"].iloc[-1],1), "m")

Points : 899
Longueur : 8978.8 m


## 2. Configuration

In [3]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]

FOREST_TAGS = [
    ("natural","wood"),
    ("landuse","forest"),
    ("landcover","trees"),
]

LOCAL_BBOX_MARGIN_DEG = 0.003
CONTROL_POINT_SPACING_M = 200.0

CORRIDOR_RADIUS_M = 10.0
FOREST_COVERAGE_THRESHOLD = 0.50

RESCUE_TRIGGER_COVERAGE = 0.10
RESCUE_MARGINS_DEG = [0.03, 0.08, 0.15]

MAX_RETRIES_PER_ENDPOINT = 2
RETRY_DELAYS_S = [2,5]

CACHE_DIR = Path("data/cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

## 3. Client Overpass avec cache et fallback serveur

In [4]:
TEMPORARY_HTTP_CODES = {429,502,503,504}
HEADERS = {"User-Agent":"PIR-Route-Agent/0.4 educational-research"}

def fetch_overpass_cached(query, cache_label):
    digest = hashlib.sha256(query.encode("utf-8")).hexdigest()[:16]
    cache_path = CACHE_DIR / f"{cache_label}_{digest}.json"

    if cache_path.exists():
        with cache_path.open("r",encoding="utf-8") as f:
            return json.load(f), "cache"

    errors = []
    for endpoint in OVERPASS_ENDPOINTS:
        for attempt in range(MAX_RETRIES_PER_ENDPOINT):
            try:
                r = requests.post(
                    endpoint,
                    data={"data":query},
                    headers=HEADERS,
                    timeout=120,
                )
                if r.status_code == 200:
                    data = r.json()
                    with cache_path.open("w",encoding="utf-8") as f:
                        json.dump(data,f,ensure_ascii=False)
                    print("Succès :", endpoint, "éléments =", len(data.get("elements",[])))
                    return data, endpoint

                body = r.text[:250].replace("\\n"," ")
                if r.status_code in TEMPORARY_HTTP_CODES:
                    errors.append(f"{endpoint} HTTP {r.status_code}: {body}")
                    if attempt < MAX_RETRIES_PER_ENDPOINT-1:
                        time.sleep(RETRY_DELAYS_S[min(attempt,len(RETRY_DELAYS_S)-1)])
                    continue
                raise RuntimeError(f"Overpass HTTP {r.status_code}: {body}")

            except requests.RequestException as exc:
                errors.append(f"{endpoint}: {exc}")
                if attempt < MAX_RETRIES_PER_ENDPOINT-1:
                    time.sleep(RETRY_DELAYS_S[min(attempt,len(RETRY_DELAYS_S)-1)])

    raise RuntimeError("Aucune instance Overpass n'a répondu.\\n" + "\\n".join(errors[-10:]))

## 4. Requêtes bbox et `is_in`

`is_in(lat,lon)` sert ici à trouver les surfaces qui **contiennent** un point du GPX, même si leur frontière est loin.

Puis `pivot` récupère le way/relation OSM source et sa géométrie complète.

In [ ]:
def forest_filters_for_bbox(bbox):
    lines = []
    for key,value in FOREST_TAGS:
        lines.append(f'way["{key}"="{value}"]({bbox});')
        lines.append(f'relation["{key}"="{value}"]({bbox});')
    return "\\n".join(lines)

def build_bbox_query(margin_deg):
    south = float(route["lat"].min() - margin_deg)
    north = float(route["lat"].max() + margin_deg)
    west  = float(route["lon"].min() - margin_deg)
    east  = float(route["lon"].max() + margin_deg)
    bbox = f"{south},{west},{north},{east}"
    return (
        "[out:json][timeout:120];\\n(\\n"
        + forest_filters_for_bbox(bbox)
        + "\\n);\\nout tags geom;"
    )

def make_control_points(spacing_m=200.0):
    d = route["distance_m"].to_numpy(dtype=float)
    lat = route["lat"].to_numpy(dtype=float)
    lon = route["lon"].to_numpy(dtype=float)

    sample_d = np.arange(0.0, float(d[-1]) + spacing_m, spacing_m)
    sample_d = np.unique(np.clip(sample_d,0.0,float(d[-1])))

    return [
        {
            "distance_m":float(x),
            "lat":float(np.interp(x,d,lat)),
            "lon":float(np.interp(x,d,lon)),
        }
        for x in sample_d
    ]

def build_is_in_query(points):
    lines = ["[out:json][timeout:120];"]
    names = []

    for i,p in enumerate(points):
        name = f"p{i}"
        names.append(name)
        lines.append(f'is_in({p["lat"]:.8f},{p["lon"]:.8f})->.{name};')

    lines.append("(" + "".join(f".{n};" for n in names) + ")->.allcontainers;")

    area_filters = []
    way_filters = []
    for key,value in FOREST_TAGS:
        area_filters.append(f'area.allcontainers["{key}"="{value}"];')
        way_filters.append(f'way.allcontainers["{key}"="{value}"];')

    lines.append("(" + "".join(area_filters) + ")->.forestareas;")
    lines.append("(" + "".join(way_filters) + ")->.forestclosedways;")

    lines.append(
        "(.forestclosedways;"
        "way(pivot.forestareas);"
        "rel(pivot.forestareas););"
    )
    lines.append("out tags geom;")
    return "\\n".join(lines)

## 5. Exécuter bbox locale + is_in

In [9]:
control_points = make_control_points(CONTROL_POINT_SPACING_M)
print("Points de contrôle :", len(control_points))

local_query = build_bbox_query(LOCAL_BBOX_MARGIN_DEG)
isin_query = build_is_in_query(control_points)

local_data, local_source = fetch_overpass_cached(local_query, "forest_bbox_local")
isin_data, isin_source = fetch_overpass_cached(isin_query, "forest_isin")

print("BBox locale :", len(local_data.get("elements",[])))
print("is_in/pivot :", len(isin_data.get("elements",[])))

Points de contrôle : 46


RuntimeError: Overpass HTTP 400: <?xml version="1.0" encoding="UTF-8"?>
<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN"
    "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">
<html xmlns="http://www.w3.org/1999/xhtml" xml:lang="en" lang="en">
<head>
  <meta http-equiv="co

## 6. Dédupliquer les objets OSM

In [7]:
def merge_osm_datasets(*datasets):
    merged = {}
    for data in datasets:
        for e in data.get("elements",[]):
            key = (e.get("type"),e.get("id"))
            if None not in key:
                merged[key] = e
    return {"elements":list(merged.values())}

osm_data = merge_osm_datasets(local_data, isin_data)
print("Objets uniques :", len(osm_data["elements"]))

NameError: name 'local_data' is not defined

## 7. Reconstruire les polygones

In [ ]:
def coords_from_geometry(geometry):
    if not geometry:
        return []
    return [
        (float(p["lon"]),float(p["lat"]))
        for p in geometry
        if "lon" in p and "lat" in p
    ]

def polygon_from_way_geometry(geometry):
    coords = coords_from_geometry(geometry)
    if len(coords) < 4 or coords[0] != coords[-1]:
        return None
    poly = Polygon(coords)
    if not poly.is_valid:
        poly = poly.buffer(0)
    return poly if not poly.is_empty else None

def polygon_from_relation(element):
    outer_lines, inner_lines = [], []

    for m in element.get("members",[]):
        coords = coords_from_geometry(m.get("geometry"))
        if len(coords) < 2:
            continue
        line = LineString(coords)
        if m.get("role","") == "inner":
            inner_lines.append(line)
        else:
            outer_lines.append(line)

    if not outer_lines:
        return None

    outer_polys = list(polygonize(unary_union(outer_lines)))
    if not outer_polys:
        return None

    result = unary_union(outer_polys)

    if inner_lines:
        inner_polys = list(polygonize(unary_union(inner_lines)))
        if inner_polys:
            result = result.difference(unary_union(inner_polys))

    if not result.is_valid:
        result = result.buffer(0)

    return result if not result.is_empty else None

def build_forest_union(dataset):
    geoms, rejected = [], []

    for e in dataset.get("elements",[]):
        geom = None

        if e.get("type") == "way":
            geom = polygon_from_way_geometry(e.get("geometry"))
        elif e.get("type") == "relation":
            geom = polygon_from_relation(e)

        if geom is None or geom.is_empty:
            rejected.append((e.get("type"),e.get("id")))
        else:
            geoms.append(geom)

    if not geoms:
        return None, geoms, rejected

    merged = unary_union(geoms)
    if not merged.is_valid:
        merged = merged.buffer(0)

    return merged, geoms, rejected

## 8. Couverture ponctuelle après bbox + is_in

In [ ]:
def point_coverage_ratio(route_df, forest_geom):
    if forest_geom is None or forest_geom.is_empty:
        return 0.0, np.zeros(len(route_df),dtype=bool)

    mask = np.array([
        forest_geom.covers(Point(float(lon),float(lat)))
        for lon,lat in zip(route_df["lon"],route_df["lat"])
    ],dtype=bool)

    return float(mask.mean()), mask

forest_wgs84, forest_geometries, rejected = build_forest_union(osm_data)
coverage_ratio, inside_mask = point_coverage_ratio(route, forest_wgs84)

print("Géométries reconstruites :", len(forest_geometries))
print("Objets rejetés :", len(rejected))
print("Points GPX couverts :", int(inside_mask.sum()), "/", len(route))
print("Couverture après bbox + is_in :", round(100*coverage_ratio,1), "%")

## 9. Fallback bbox élargi

Overpass ne crée pas forcément une `area` pour toutes les relations OSM.
Si la couverture reste < 10 %, on élargit progressivement la bbox.

In [ ]:
rescue_datasets = []

if coverage_ratio < RESCUE_TRIGGER_COVERAGE:
    print("Couverture faible : fallback élargi.")

    for margin in RESCUE_MARGINS_DEG:
        print("\nMarge :", margin, "°")

        rescue_query = build_bbox_query(margin)
        rescue_data, rescue_source = fetch_overpass_cached(
            rescue_query,
            f"forest_bbox_rescue_{str(margin).replace('.','_')}",
        )
        rescue_datasets.append(rescue_data)

        combined = merge_osm_datasets(local_data, isin_data, *rescue_datasets)
        candidate_forest, candidate_geoms, candidate_rejected = build_forest_union(combined)
        candidate_ratio, candidate_mask = point_coverage_ratio(route, candidate_forest)

        print("Couverture :", round(100*candidate_ratio,1), "%")

        if candidate_ratio >= coverage_ratio:
            osm_data = combined
            forest_wgs84 = candidate_forest
            forest_geometries = candidate_geoms
            rejected = candidate_rejected
            coverage_ratio = candidate_ratio
            inside_mask = candidate_mask

        if coverage_ratio >= RESCUE_TRIGGER_COVERAGE:
            break

print("\nCOUVERTURE OSM FINALE :", round(100*coverage_ratio,1), "%")

## 10. Diagnostic tags/types

In [ ]:
tag_counts = Counter()
type_counts = Counter()

for e in osm_data.get("elements",[]):
    type_counts[e.get("type")] += 1
    tags = e.get("tags",{})
    for key,value in FOREST_TAGS:
        if tags.get(key) == value:
            tag_counts[f"{key}={value}"] += 1

print("Types :", type_counts)
print("Tags :", tag_counts)
print("Objets uniques :", len(osm_data.get("elements",[])))
print("Géométries reconstruites :", len(forest_geometries))
print("Rejets :", len(rejected))

## 11. Projection métrique

In [ ]:
def utm_epsg_for_lonlat(lon,lat):
    zone = int((lon+180)/6)+1
    return (32600+zone) if lat >= 0 else (32700+zone)

center_lon = float(route["lon"].mean())
center_lat = float(route["lat"].mean())
utm_epsg = utm_epsg_for_lonlat(center_lon,center_lat)

transformer = Transformer.from_crs(
    "EPSG:4326",
    f"EPSG:{utm_epsg}",
    always_xy=True,
)

project_geom = lambda geom: transform(transformer.transform,geom)

forest_m = project_geom(forest_wgs84) if forest_wgs84 is not None else None

print("CRS :", f"EPSG:{utm_epsg}")

## 12. Couverture forestière du corridor

In [ ]:
projected_points = [
    transformer.transform(float(lon),float(lat))
    for lon,lat in zip(route["lon"],route["lat"])
]

rows = []

for i in range(len(route)-1):
    start_m = float(route.loc[i,"distance_m"])
    end_m   = float(route.loc[i+1,"distance_m"])

    segment = LineString([projected_points[i],projected_points[i+1]])
    corridor = segment.buffer(CORRIDOR_RADIUS_M,cap_style=2)

    if forest_m is None or forest_m.is_empty or corridor.area <= 0:
        coverage = 0.0
    else:
        coverage = corridor.intersection(forest_m).area / corridor.area

    coverage = float(max(0.0,min(1.0,coverage)))

    rows.append({
        "start_m":start_m,
        "end_m":end_m,
        "forest_coverage":coverage,
        "is_forest":bool(coverage >= FOREST_COVERAGE_THRESHOLD),
    })

forest_segments = pd.DataFrame(rows)

print("Segments forêt :", int(forest_segments["is_forest"].sum()), "/", len(forest_segments))
print("Part classée forêt :", round(100*forest_segments["is_forest"].mean(),1), "%")
display(forest_segments.head(20))

## 13. Visualisation

In [ ]:
fig,ax = plt.subplots()
x = (forest_segments["start_m"] + forest_segments["end_m"]) / 2
ax.plot(x,forest_segments["forest_coverage"])
ax.axhline(FOREST_COVERAGE_THRESHOLD,linestyle="--")
ax.set_xlabel("Distance (m)")
ax.set_ylabel("Couverture forestière OSM")
ax.set_ylim(-0.05,1.05)
ax.set_title("Couverture forestière le long de l'itinéraire")
plt.show()

## 14. Fusionner les segments forestiers contigus

In [ ]:
raw_groups = []
current = None

for _,row in forest_segments.iterrows():
    if not bool(row["is_forest"]):
        if current is not None:
            raw_groups.append(current)
            current = None
        continue

    c = float(row["forest_coverage"])

    if current is None:
        current = {
            "start_m":float(row["start_m"]),
            "end_m":float(row["end_m"]),
            "coverages":[c],
        }
    elif abs(float(row["start_m"]) - current["end_m"]) < 1e-6:
        current["end_m"] = float(row["end_m"])
        current["coverages"].append(c)
    else:
        raw_groups.append(current)
        current = {
            "start_m":float(row["start_m"]),
            "end_m":float(row["end_m"]),
            "coverages":[c],
        }

if current is not None:
    raw_groups.append(current)

final_forest_annotations = []

for g in raw_groups:
    vals = g["coverages"]

    final_forest_annotations.append({
        "feature":"foret",
        "family":"terrain",
        "start_m":g["start_m"],
        "end_m":g["end_m"],
        "resolution_m":10.0,
        "confidence":float(np.mean(vals)),
        "metrics":{
            "source":"OpenStreetMap",
            "retrieval_strategy":"bbox+is_in+pivot" + ("+expanded_bbox_rescue" if rescue_datasets else ""),
            "mean_forest_coverage":float(np.mean(vals)),
            "min_forest_coverage":float(np.min(vals)),
            "max_forest_coverage":float(np.max(vals)),
            "corridor_radius_m":CORRIDOR_RADIUS_M,
            "coverage_threshold":FOREST_COVERAGE_THRESHOLD,
            "n_base_segments":len(vals),
        },
    })

print("Portions forestières :", len(final_forest_annotations))
display(pd.DataFrame(final_forest_annotations))

## 15. Export compatible avec le notebook 04

In [ ]:
output_dir = Path("data/output")
output_dir.mkdir(parents=True,exist_ok=True)

forest_json_path = output_dir / "terrain_forest_annotations.json"
forest_csv_path = output_dir / "terrain_forest_annotations.csv"

with forest_json_path.open("w",encoding="utf-8") as f:
    json.dump(final_forest_annotations,f,ensure_ascii=False,indent=2)

pd.DataFrame([
    {
        **{k:v for k,v in a.items() if k != "metrics"},
        "metrics_json":json.dumps(a["metrics"],ensure_ascii=False),
    }
    for a in final_forest_annotations
]).to_csv(forest_csv_path,index=False)

print("Export :", forest_json_path)
print("Export :", forest_csv_path)

## Diagnostic attendu

Sur votre trace test, l'ancienne version donnait environ 2,4 % de points GPX contenus dans les polygones récupérés.

Regardez maintenant :

```text
Couverture après bbox + is_in : XX %
COUVERTURE OSM FINALE : YY %
```

Si `XX` augmente fortement, le grand massif englobant a été récupéré par `is_in + pivot`.

Si `XX` reste faible mais `YY` augmente, c'est le fallback bbox élargi qui a récupéré une relation/multipolygone absente des `areas` Overpass.

Si les deux restent très faibles, la donnée OSM elle-même est probablement incomplète ou une relation très complexe est rejetée lors de la reconstruction.

# Partie B — Configuration de l'Agent IA 2

Pour le premier test :

```python
RUN_VECTOR_AGENT = True
RUN_VISION_AGENT = False
PRIMARY_OUTPUT_METHOD = "vector_agent"
```

Pour le test multimodal, installer d'abord :

```bash
ollama pull qwen3-vl:4b
```

puis passer `RUN_VISION_AGENT = True`.

`qwen3-vl:4b` est utilisé uniquement pour l'expérience visuelle.

In [ ]:
import base64

RUN_VECTOR_AGENT = True
RUN_VISION_AGENT = False

# "vector_agent", "vision_agent" ou "reference"
PRIMARY_OUTPUT_METHOD = "vector_agent"

OLLAMA_URL = "http://localhost:11434/api/chat"
TEXT_MODEL = "llama3.1:8b"
VISION_MODEL = "qwen3-vl:4b"

# L'agent vectoriel choisit librement sa granularité dans ces bornes.
# Ce sont des garde-fous, pas une liste de choix.
VECTOR_SAMPLE_BOUNDS_M = (25.0, 300.0)
VECTOR_SIMPLIFY_BOUNDS_M = (5.0, 200.0)

VISION_MARKER_SPACING_M = 500.0
VISION_IMAGE_DPI = 160

ROUTE_LENGTH_M = float(
    route["distance_m"].iloc[-1]
)

print("Longueur route :", round(ROUTE_LENGTH_M, 1), "m")

# Partie C — Référence SIG : `INTERSECT`

La référence géométrique est calculée directement avec :

```text
LineString itinéraire
INTERSECT
Polygon / MultiPolygon forêt
```

Elle servira ensuite à évaluer l'Agent IA 2.

In [ ]:
def extract_line_geometries(geometry):
    if geometry is None or geometry.is_empty:
        return []

    if geometry.geom_type == "LineString":
        return [geometry]

    if geometry.geom_type == "MultiLineString":
        return list(geometry.geoms)

    if geometry.geom_type == "GeometryCollection":
        lines = []

        for child in geometry.geoms:
            lines.extend(
                extract_line_geometries(child)
            )

        return lines

    return []


def merge_intervals(
    intervals,
    gap_tolerance_m=1.0,
):
    if not intervals:
        return []

    clean = sorted(
        [
            [
                max(0.0, float(start)),
                min(
                    ROUTE_LENGTH_M,
                    float(end),
                ),
            ]
            for start, end in intervals
            if float(end) > float(start)
        ],
        key=lambda x: x[0],
    )

    if not clean:
        return []

    merged = [clean[0]]

    for start, end in clean[1:]:
        if (
            start
            <= merged[-1][1]
            + gap_tolerance_m
        ):
            merged[-1][1] = max(
                merged[-1][1],
                end,
            )
        else:
            merged.append(
                [start, end]
            )

    return [
        (
            float(start),
            float(end),
        )
        for start, end in merged
    ]


route_distances = (
    route["distance_m"]
    .to_numpy(dtype=float)
)

projected_points_agent2 = [
    transformer.transform(
        float(lon),
        float(lat),
    )
    for lon, lat in zip(
        route["lon"],
        route["lat"],
    )
]


def reference_intersect():
    if (
        forest_m is None
        or forest_m.is_empty
    ):
        return []

    intervals = []

    for i in range(len(route) - 1):
        segment = LineString([
            projected_points_agent2[i],
            projected_points_agent2[i + 1],
        ])

        if segment.length <= 0:
            continue

        intersection = (
            segment.intersection(
                forest_m
            )
        )

        for line in extract_line_geometries(
            intersection
        ):
            if line.length <= 0:
                continue

            p0 = Point(
                line.coords[0]
            )

            p1 = Point(
                line.coords[-1]
            )

            f0 = (
                segment.project(p0)
                / segment.length
            )

            f1 = (
                segment.project(p1)
                / segment.length
            )

            d0 = float(
                route_distances[i]
            )

            d1 = float(
                route_distances[i + 1]
            )

            start = (
                d0
                + min(f0, f1)
                * (d1 - d0)
            )

            end = (
                d0
                + max(f0, f1)
                * (d1 - d0)
            )

            intervals.append(
                (start, end)
            )

    return merge_intervals(
        intervals,
        gap_tolerance_m=1.0,
    )


reference_intervals = reference_intersect()

reference_length_m = sum(
    end - start
    for start, end
    in reference_intervals
)

print(
    "Portions forêt INTERSECT :",
    len(reference_intervals),
)

print(
    "Longueur en forêt :",
    round(reference_length_m, 1),
    "m",
)

print(
    "Part de l'itinéraire :",
    round(
        100
        * reference_length_m
        / max(
            ROUTE_LENGTH_M,
            1e-9,
        ),
        1,
    ),
    "%",
)

display(
    pd.DataFrame(
        reference_intervals,
        columns=[
            "start_m",
            "end_m",
        ],
    )
)

# Partie D — Client Ollama pour les agents

In [ ]:
def extract_json_object(text):
    text = text.strip()

    try:
        return json.loads(text)

    except json.JSONDecodeError:
        start = text.find("{")
        end = text.rfind("}")

        if (
            start == -1
            or end == -1
            or end <= start
        ):
            raise

        return json.loads(
            text[start:end + 1]
        )


def ask_ollama_json(
    model,
    prompt,
    images_base64=None,
):
    message = {
        "role": "user",
        "content": prompt,
    }

    if images_base64:
        message["images"] = (
            images_base64
        )

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": model,
            "messages": [message],
            "stream": False,
            "format": "json",
            "options": {
                "temperature": 0,
            },
        },
        timeout=180,
    )

    if response.status_code != 200:
        raise RuntimeError(
            f"Ollama HTTP "
            f"{response.status_code}: "
            f"{response.text}"
        )

    content = (
        response
        .json()["message"]["content"]
    )

    return extract_json_object(
        content
    )


def normalize_agent_intervals(
    raw_intervals,
):
    clean = []

    for item in raw_intervals:
        try:
            start = float(
                item["start_m"]
            )

            end = float(
                item["end_m"]
            )

            confidence = float(
                item.get(
                    "confidence",
                    0.5,
                )
            )

        except (
            KeyError,
            TypeError,
            ValueError,
        ):
            continue

        start = max(
            0.0,
            min(
                start,
                ROUTE_LENGTH_M,
            ),
        )

        end = max(
            0.0,
            min(
                end,
                ROUTE_LENGTH_M,
            ),
        )

        if end <= start:
            continue

        clean.append({
            "start_m": start,
            "end_m": end,
            "confidence": min(
                1.0,
                max(
                    0.0,
                    confidence,
                ),
            ),
        })

    clean = sorted(
        clean,
        key=lambda x: x[
            "start_m"
        ],
    )

    merged = []

    for item in clean:
        if (
            not merged
            or item["start_m"]
            > merged[-1]["end_m"]
            + 10.0
        ):
            merged.append(
                dict(item)
            )

        else:
            previous = merged[-1]

            previous["end_m"] = max(
                previous["end_m"],
                item["end_m"],
            )

            previous["confidence"] = (
                previous["confidence"]
                + item["confidence"]
            ) / 2.0

    return merged

# Partie E — Méthode 1 : Agent IA vectoriel

L'agent ne reçoit pas la sortie de `INTERSECT`.

Il reçoit uniquement :

- la trace GPX échantillonnée ;
- les polygones OSM ;
- les tags OSM ;
- la longueur totale de l'itinéraire.

Avant l'analyse, il choisit lui-même la granularité de la représentation vectorielle.

In [ ]:
def choose_vector_representation():
    prompt = f'''
Tu es un agent IA chargé d'analyser spatialement un itinéraire
et des polygones OpenStreetMap.

Objectif :
déterminer quelles portions de l'itinéraire traversent une zone
que l'on peut qualifier sémantiquement de forêt.

Longueur de l'itinéraire :
{ROUTE_LENGTH_M:.1f} m

Objets OSM récupérés :
{len(osm_data.get("elements", []))}

Tags OSM :
{dict(tag_counts)}

Choisis librement la granularité de la représentation que tu veux examiner.

Contraintes de sécurité uniquement :
- route_sample_spacing_m entre {VECTOR_SAMPLE_BOUNDS_M[0]} et {VECTOR_SAMPLE_BOUNDS_M[1]} m ;
- polygon_simplify_m entre {VECTOR_SIMPLIFY_BOUNDS_M[0]} et {VECTOR_SIMPLIFY_BOUNDS_M[1]} m.

Une faible valeur conserve davantage de détails.
Une grande valeur réduit le nombre de coordonnées.

Réponds uniquement en JSON :
{{
  "route_sample_spacing_m": 100.0,
  "polygon_simplify_m": 30.0,
  "justification": "courte justification"
}}
'''

    response = ask_ollama_json(
        TEXT_MODEL,
        prompt,
    )

    route_spacing = float(
        response[
            "route_sample_spacing_m"
        ]
    )

    polygon_simplify = float(
        response[
            "polygon_simplify_m"
        ]
    )

    route_spacing = min(
        max(
            route_spacing,
            VECTOR_SAMPLE_BOUNDS_M[0],
        ),
        VECTOR_SAMPLE_BOUNDS_M[1],
    )

    polygon_simplify = min(
        max(
            polygon_simplify,
            VECTOR_SIMPLIFY_BOUNDS_M[0],
        ),
        VECTOR_SIMPLIFY_BOUNDS_M[1],
    )

    return {
        "route_sample_spacing_m": route_spacing,
        "polygon_simplify_m": polygon_simplify,
        "justification": str(
            response.get(
                "justification",
                "",
            )
        ),
    }


def sample_route_for_agent(
    spacing_m,
):
    d = route[
        "distance_m"
    ].to_numpy(dtype=float)

    lat = route[
        "lat"
    ].to_numpy(dtype=float)

    lon = route[
        "lon"
    ].to_numpy(dtype=float)

    samples = np.arange(
        0.0,
        ROUTE_LENGTH_M
        + spacing_m,
        spacing_m,
    )

    samples = np.unique(
        np.clip(
            samples,
            0.0,
            ROUTE_LENGTH_M,
        )
    )

    return [
        {
            "distance_m": round(
                float(distance),
                1,
            ),
            "lat": round(
                float(
                    np.interp(
                        distance,
                        d,
                        lat,
                    )
                ),
                7,
            ),
            "lon": round(
                float(
                    np.interp(
                        distance,
                        d,
                        lon,
                    )
                ),
                7,
            ),
        }
        for distance in samples
    ]


inverse_transformer = Transformer.from_crs(
    f"EPSG:{utm_epsg}",
    "EPSG:4326",
    always_xy=True,
)


def build_vector_payload(
    decision,
):
    route_samples = (
        sample_route_for_agent(
            decision[
                "route_sample_spacing_m"
            ]
        )
    )

    if forest_m is None:
        forest_geojson = None

    else:
        simplified_m = (
            forest_m.simplify(
                decision[
                    "polygon_simplify_m"
                ],
                preserve_topology=True,
            )
        )

        simplified_wgs84 = (
            transform(
                inverse_transformer.transform,
                simplified_m,
            )
        )

        forest_geojson = mapping(
            simplified_wgs84
        )

    return {
        "route_length_m": round(
            ROUTE_LENGTH_M,
            1,
        ),
        "route_samples": route_samples,
        "forest_tags_summary": dict(
            tag_counts
        ),
        "forest_geometry": (
            forest_geojson
        ),
    }

In [ ]:
def run_vector_agent():
    decision = (
        choose_vector_representation()
    )

    print(
        "Granularité choisie :",
        decision,
    )

    payload = build_vector_payload(
        decision
    )

    payload_json = json.dumps(
        payload,
        ensure_ascii=False,
        separators=(",", ":"),
    )

    print(
        "Taille paquet vectoriel :",
        len(payload_json),
        "caractères",
    )

    prompt = f'''
Tu es l'Agent IA 2 de reconnaissance environnementale.

Tu reçois une trace d'itinéraire échantillonnée et des polygones OSM
représentant des zones boisées.

Ta tâche est double.

1. Interprétation sémantique :
détermine si les tags et polygones fournis peuvent être normalisés
sous l'annotation PIR "foret".

2. Interprétation spatiale :
estime les portions de la trace situées à l'intérieur de ces polygones.

Important :
- utilise uniquement les données vectorielles fournies ;
- tu n'as PAS accès au résultat de l'algorithme INTERSECT ;
- respecte les distances curvilignes fournies ;
- ne prétends pas à une précision supérieure à l'échantillonnage ;
- les intervalles doivent être compris entre 0 et {ROUTE_LENGTH_M:.1f} m.

DONNEES :
{payload_json}

Réponds uniquement en JSON :
{{
  "semantic_label": "foret",
  "semantic_confidence": 0.9,
  "semantic_reason": "courte justification",
  "intervals": [
    {{
      "start_m": 1200.0,
      "end_m": 2500.0,
      "confidence": 0.8
    }}
  ]
}}
'''

    response = ask_ollama_json(
        TEXT_MODEL,
        prompt,
    )

    return {
        "representation": decision,
        "semantic_label": response.get(
            "semantic_label",
            "inconnu",
        ),
        "semantic_confidence": float(
            response.get(
                "semantic_confidence",
                0.0,
            )
        ),
        "semantic_reason": str(
            response.get(
                "semantic_reason",
                "",
            )
        ),
        "intervals": (
            normalize_agent_intervals(
                response.get(
                    "intervals",
                    [],
                )
            )
        ),
    }


vector_agent_result = None

if RUN_VECTOR_AGENT:
    vector_agent_result = (
        run_vector_agent()
    )

    print(
        json.dumps(
            vector_agent_result,
            ensure_ascii=False,
            indent=2,
        )
    )

else:
    print(
        "Agent vectoriel désactivé."
    )

# Partie F — Méthode 2 : Agent IA visuel

On transforme la même scène en image :

```text
polygones forêt
+
itinéraire
+
repères tous les 500 m
```

Le modèle multimodal doit estimer les portions forestières sans recevoir
la sortie `INTERSECT`.

In [ ]:
VISION_IMAGE_PATH = (
    output_dir
    / "agent2_forest_visual_context.png"
)


def draw_polygon(
    ax,
    geometry,
):
    if (
        geometry is None
        or geometry.is_empty
    ):
        return

    if geometry.geom_type == "Polygon":
        x, y = geometry.exterior.xy

        ax.fill(
            x,
            y,
            alpha=0.25,
        )

    elif geometry.geom_type == "MultiPolygon":
        for polygon in geometry.geoms:
            draw_polygon(
                ax,
                polygon,
            )


def build_visual_context():
    fig, ax = plt.subplots(
        figsize=(10, 10)
    )

    draw_polygon(
        ax,
        forest_m,
    )

    route_line_agent = LineString(
        projected_points_agent2
    )

    x_route, y_route = (
        route_line_agent.xy
    )

    ax.plot(
        x_route,
        y_route,
        linewidth=2.5,
        label="Itinéraire",
    )

    xs = np.array(
        [
            point[0]
            for point in projected_points_agent2
        ]
    )

    ys = np.array(
        [
            point[1]
            for point in projected_points_agent2
        ]
    )

    marker_distances = np.arange(
        0.0,
        ROUTE_LENGTH_M
        + VISION_MARKER_SPACING_M,
        VISION_MARKER_SPACING_M,
    )

    marker_distances = np.unique(
        np.clip(
            marker_distances,
            0.0,
            ROUTE_LENGTH_M,
        )
    )

    for distance in marker_distances:
        x = float(
            np.interp(
                distance,
                route_distances,
                xs,
            )
        )

        y = float(
            np.interp(
                distance,
                route_distances,
                ys,
            )
        )

        ax.scatter(
            [x],
            [y],
            s=18,
        )

        ax.annotate(
            f"{distance/1000:.1f} km",
            (x, y),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

    ax.set_aspect(
        "equal",
        adjustable="datalim",
    )

    ax.set_title(
        "Itinéraire et zones boisées OSM"
    )

    ax.set_xlabel(
        "Coordonnée métrique X"
    )

    ax.set_ylabel(
        "Coordonnée métrique Y"
    )

    ax.legend()

    fig.tight_layout()

    fig.savefig(
        VISION_IMAGE_PATH,
        dpi=VISION_IMAGE_DPI,
        bbox_inches="tight",
    )

    plt.show()

    print(
        "Image créée :",
        VISION_IMAGE_PATH,
    )


build_visual_context()

In [ ]:
def ollama_model_names():
    try:
        response = requests.get(
            "http://localhost:11434/api/tags",
            timeout=20,
        )

        if response.status_code != 200:
            return []

        return [
            model.get(
                "name",
                "",
            )
            for model
            in response.json().get(
                "models",
                [],
            )
        ]

    except requests.RequestException:
        return []


def image_to_base64(path):
    with path.open("rb") as f:
        return base64.b64encode(
            f.read()
        ).decode("ascii")


def run_vision_agent():
    installed = (
        ollama_model_names()
    )

    if not any(
        name.startswith(
            VISION_MODEL.split(":")[0]
        )
        for name in installed
    ):
        raise RuntimeError(
            f"Le modèle {VISION_MODEL} "
            "n'est pas installé. Exécute : "
            f"ollama pull {VISION_MODEL}"
        )

    image_b64 = image_to_base64(
        VISION_IMAGE_PATH
    )

    prompt = f'''
Tu es l'Agent IA 2 multimodal de reconnaissance environnementale.

L'image contient :
- un itinéraire ;
- des zones boisées OpenStreetMap ;
- des repères de distance le long de la trace.

Longueur totale :
{ROUTE_LENGTH_M:.1f} m

Tags OSM associés :
{dict(tag_counts)}

Ta tâche :
1. décider si les zones visibles peuvent être qualifiées de "foret" ;
2. observer où l'itinéraire traverse visuellement ces zones ;
3. estimer les intervalles start_m / end_m en utilisant les repères.

Tu n'as PAS accès à l'intersection géométrique calculée par Python.
Ne prétends pas à une précision supérieure à ce que l'image permet.

Réponds uniquement en JSON :
{{
  "semantic_label": "foret",
  "semantic_confidence": 0.9,
  "semantic_reason": "courte justification",
  "intervals": [
    {{
      "start_m": 1000.0,
      "end_m": 2500.0,
      "confidence": 0.7
    }}
  ]
}}
'''

    response = ask_ollama_json(
        VISION_MODEL,
        prompt,
        images_base64=[
            image_b64
        ],
    )

    return {
        "semantic_label": response.get(
            "semantic_label",
            "inconnu",
        ),
        "semantic_confidence": float(
            response.get(
                "semantic_confidence",
                0.0,
            )
        ),
        "semantic_reason": str(
            response.get(
                "semantic_reason",
                "",
            )
        ),
        "intervals": (
            normalize_agent_intervals(
                response.get(
                    "intervals",
                    [],
                )
            )
        ),
    }


vision_agent_result = None

if RUN_VISION_AGENT:
    vision_agent_result = (
        run_vision_agent()
    )

    print(
        json.dumps(
            vision_agent_result,
            ensure_ascii=False,
            indent=2,
        )
    )

else:
    print(
        "Agent visuel désactivé. "
        "Pour le tester : "
        f"ollama pull {VISION_MODEL}, "
        "puis RUN_VISION_AGENT=True."
    )

# Partie G — Évaluer les deux méthodes contre `INTERSECT`

La référence n'est utilisée **qu'après** les prédictions.

In [ ]:
def intervals_from_agent(
    result,
):
    if result is None:
        return []

    return [
        (
            float(item["start_m"]),
            float(item["end_m"]),
        )
        for item in result.get(
            "intervals",
            [],
        )
    ]


def interval_union_length(
    intervals,
):
    merged = merge_intervals(
        intervals,
        gap_tolerance_m=0.0,
    )

    return sum(
        end - start
        for start, end in merged
    )


def interval_intersection_length(
    intervals_a,
    intervals_b,
):
    a = merge_intervals(
        intervals_a,
        gap_tolerance_m=0.0,
    )

    b = merge_intervals(
        intervals_b,
        gap_tolerance_m=0.0,
    )

    total = 0.0

    for a_start, a_end in a:
        for b_start, b_end in b:
            total += max(
                0.0,
                min(
                    a_end,
                    b_end,
                )
                - max(
                    a_start,
                    b_start,
                ),
            )

    return total


def evaluate_intervals(
    predicted,
    reference,
):
    predicted_length = (
        interval_union_length(
            predicted
        )
    )

    reference_length = (
        interval_union_length(
            reference
        )
    )

    intersection = (
        interval_intersection_length(
            predicted,
            reference,
        )
    )

    union = (
        predicted_length
        + reference_length
        - intersection
    )

    return {
        "predicted_length_m": float(
            predicted_length
        ),
        "reference_length_m": float(
            reference_length
        ),
        "intersection_length_m": float(
            intersection
        ),
        "precision_length": float(
            intersection
            / predicted_length
            if predicted_length > 0
            else 0.0
        ),
        "recall_length": float(
            intersection
            / reference_length
            if reference_length > 0
            else 0.0
        ),
        "iou_linear": float(
            intersection
            / union
            if union > 0
            else 1.0
        ),
    }


evaluation = {}

if vector_agent_result is not None:
    evaluation[
        "vector_agent"
    ] = evaluate_intervals(
        intervals_from_agent(
            vector_agent_result
        ),
        reference_intervals,
    )

if vision_agent_result is not None:
    evaluation[
        "vision_agent"
    ] = evaluate_intervals(
        intervals_from_agent(
            vision_agent_result
        ),
        reference_intervals,
    )

print(
    json.dumps(
        evaluation,
        ensure_ascii=False,
        indent=2,
    )
)

# Partie H — Export compatible avec le notebook 04

Toutes les méthodes sont sauvegardées séparément.

`PRIMARY_OUTPUT_METHOD` choisit celle qui alimente réellement le notebook 04.

In [ ]:
def intervals_to_annotations(
    intervals,
    method,
    resolution_m,
):
    annotations = []

    for item in intervals:
        if isinstance(item, dict):
            start = float(
                item["start_m"]
            )

            end = float(
                item["end_m"]
            )

            confidence = float(
                item.get(
                    "confidence",
                    0.8,
                )
            )

        else:
            start = float(
                item[0]
            )

            end = float(
                item[1]
            )

            confidence = 1.0

        annotations.append({
            "feature": "foret",
            "family": "terrain",
            "start_m": start,
            "end_m": end,
            "resolution_m": float(
                resolution_m
            ),
            "confidence": confidence,
            "metrics": {
                "source": "OpenStreetMap",
                "method": method,
            },
        })

    return annotations


reference_annotations = (
    intervals_to_annotations(
        reference_intervals,
        "python_intersect_reference",
        10.0,
    )
)

vector_annotations = []

if vector_agent_result is not None:
    vector_annotations = (
        intervals_to_annotations(
            vector_agent_result[
                "intervals"
            ],
            "llm_vector_agent",
            vector_agent_result[
                "representation"
            ][
                "route_sample_spacing_m"
            ],
        )
    )

vision_annotations = []

if vision_agent_result is not None:
    vision_annotations = (
        intervals_to_annotations(
            vision_agent_result[
                "intervals"
            ],
            "llm_vision_agent",
            VISION_MARKER_SPACING_M,
        )
    )


method_outputs = {
    "reference": reference_annotations,
    "vector_agent": vector_annotations,
    "vision_agent": vision_annotations,
}

if (
    PRIMARY_OUTPUT_METHOD
    not in method_outputs
):
    raise ValueError(
        "PRIMARY_OUTPUT_METHOD invalide."
    )

if (
    PRIMARY_OUTPUT_METHOD
    == "vector_agent"
    and not RUN_VECTOR_AGENT
):
    raise RuntimeError(
        "vector_agent est choisi "
        "mais RUN_VECTOR_AGENT=False."
    )

if (
    PRIMARY_OUTPUT_METHOD
    == "vision_agent"
    and not RUN_VISION_AGENT
):
    raise RuntimeError(
        "vision_agent est choisi "
        "mais RUN_VISION_AGENT=False."
    )

primary_annotations = (
    method_outputs[
        PRIMARY_OUTPUT_METHOD
    ]
)

reference_path = (
    output_dir
    / "terrain_forest_reference_intersect.json"
)

vector_path = (
    output_dir
    / "terrain_forest_agent_vector.json"
)

vision_path = (
    output_dir
    / "terrain_forest_agent_vision.json"
)

primary_path = (
    output_dir
    / "terrain_forest_annotations.json"
)

evaluation_path = (
    output_dir
    / "agent2_forest_evaluation.json"
)

for path, data in [
    (
        reference_path,
        reference_annotations,
    ),
    (
        vector_path,
        vector_annotations,
    ),
    (
        vision_path,
        vision_annotations,
    ),
    (
        primary_path,
        primary_annotations,
    ),
]:
    with path.open(
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

with evaluation_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        {
            "primary_output_method": (
                PRIMARY_OUTPUT_METHOD
            ),
            "reference_intervals": (
                reference_intervals
            ),
            "vector_agent_result": (
                vector_agent_result
            ),
            "vision_agent_result": (
                vision_agent_result
            ),
            "evaluation": evaluation,
        },
        f,
        ensure_ascii=False,
        indent=2,
    )

print(
    "Méthode envoyée au notebook 04 :",
    PRIMARY_OUTPUT_METHOD,
)

print("Exports :")
print("-", reference_path)
print("-", vector_path)
print("-", vision_path)
print("-", primary_path)
print("-", evaluation_path)

# Lecture scientifique de cette V2

Le notebook 03bis n'est plus seulement :

```text
Overpass
→ polygones
→ calcul Python
```

Il devient :

```text
données OSM
      ↓
Agent IA 2
      ↓
choix de représentation
      ↓
interprétation sémantique + spatiale
      ↓
portions forêt
```

avec deux modalités expérimentales :

```text
1. vecteur
2. vision
```

La méthode déterministe `INTERSECT` reste à côté comme **référence indépendante**.

### Ce qu'il faudra noter pendant les tests

Pour chaque GPX :

- couverture forêt `INTERSECT` ;
- résultat Agent vectoriel ;
- granularité choisie par l'Agent vectoriel ;
- `precision_length` ;
- `recall_length` ;
- `IoU linéaire` ;
- résultat Agent visuel lorsqu'il est activé ;
- différences qualitatives observées.

Cette expérience pourra ensuite être reproduite sur la pente.